# Infosys (BSE: 500209) — Reliable Financial Data & FCFF Valuation

## Beginner Hinglish guide
Is notebook mein pehle saved official-report records validate hote hain; agar critical data, reconciliation ya dated share denominator missing ho, valuation ruk jayegi. Amounts INR crore mein hain aur shares crore shares mein, isliye `crore / crore = INR per share`. Yeh educational DCF output hai, guaranteed fair value nahi.

## Method
Operating EBIT = PBT + finance cost − other income. Operating NWC = trade receivables − trade payables. FCFF = EBIT × (1 − operating tax) + D&A − capex − change in NWC. Cash and investments are not automatically excess cash; the bridge explicitly deducts leases and NCI.

## Reconciliation & share denominator
Har FY ke liye assets = equity + liabilities, cash-flow subtotal, opening-to-closing cash roll-forward, aur CFS-to-balance-sheet cash comparison check hota hai. FY26 valuation denominator weighted-average diluted EPS shares nahi hai: it is 31 March 2026 paid-up shares minus disclosed treasury shares. Employee-option dilution is a visible limitation.

In [1]:
from pathlib import Path
import sys, pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from equity_platform import build_historical, load_assumptions, reconcile_statements, run_fcff_valuation, sensitivity, validate_source_records, valuation_readiness
from market_data import market_comparison, select_share_observation
records = pd.read_csv(ROOT / 'data/raw/infosys_source_records.csv')
errors = validate_source_records(records)
assert not errors, 'Validation blocks valuation: ' + '; '.join(errors)
shares = pd.read_csv(ROOT / 'data/shares/infosys_share_observations.csv')
historical = build_historical(records)
historical[['fiscal_year','revenue','operating_ebit','operating_nwc','change_operating_nwc','shares_crore']]


field,fiscal_year,revenue,operating_ebit,operating_nwc,change_operating_nwc,shares_crore
0,2024,153670.0,31747.0,26237.0,NaN,414.468043
1,2025,162990.0,34424.0,26994.0,757.0,415.205118
2,2026,178650.0,36089.0,30490.0,3496.0,412.010817


In [2]:
# Reconciliation is a hard gate: no balancing plug and no valuation if a check fails.
reconciliation = reconcile_statements(historical)
display(reconciliation)
assumptions = load_assumptions(ROOT / 'config/assumptions.json')
valuation_readiness(historical, reconciliation, shares, 'infosys', assumptions['valuation_date'])


,fiscal_year,check,status,residual_inr_crore,tolerance_inr_crore,reason
0,2024,assets = equity + liabilities,pass,0.0,1.0,Rounded INR-crore statement comparison.
1,2024,operating + investing + financing = reported n...,pass,0.0,1.0,Rounded INR-crore statement comparison.
2,2024,opening cash + net cash change + FX = closing ...,pass,0.0,1.0,Rounded INR-crore statement comparison.
3,2024,cash-flow closing cash equivalents = balance-s...,pass,0.0,1.0,Restricted cash is separately reported and exc...
4,2025,assets = equity + liabilities,pass,0.0,1.0,Rounded INR-crore statement comparison.
5,2025,operating + investing + financing = reported n...,pass,0.0,1.0,Rounded INR-crore statement comparison.
6,2025,opening cash + net cash change + FX = closing ...,pass,0.0,1.0,Rounded INR-crore statement comparison.
7,2025,cash-flow closing cash equivalents = balance-s...,pass,0.0,1.0,Restricted cash is separately reported and exc...
8,2026,assets = equity + liabilities,pass,0.0,1.0,Rounded INR-crore statement comparison.
9,2026,operating + investing + financing = reported n...,pass,0.0,1.0,Rounded INR-crore statement comparison.


,area,status,detail
0,financial_data_validation,pass,Source-record validation passed.
1,statement_reconciliation,pass,All required statement checks must pass before...
2,share_denominator,warning,"Period-end net shares are verified, but fully ..."
3,market_comparison,unavailable,A direct BSE close on the valuation/share date...


In [3]:
forecast, summary = run_fcff_valuation(historical, assumptions, shares, 'infosys')
display(pd.Series(summary, name='INR crore except per share and shares'))
market = pd.read_csv(ROOT / 'data/market/infosys_market_observations.csv')
display(pd.Series(market_comparison(summary, market, select_share_observation(shares, 'infosys', assumptions['valuation_date']), 'infosys'), name='Market-comparison status'))
sensitivity(historical, assumptions, [0.10, 0.12, 0.14], [0.04, 0.05, 0.06], shares, 'infosys').round(2)


valuation_date                                                            2026-06-30
information_cutoff                                                        2026-07-23
valuation_status                                   provisional_period_end_net_shares
pv_forecast_fcff_inr_crore                                             122206.071307
pv_terminal_value_inr_crore                                            333946.551094
enterprise_value_inr_crore                                               456152.6224
equity_value_inr_crore                                                   446531.6224
shares_crore                                                              404.964581
value_per_share_inr                                                      1102.643646
share_denominator_method           Period-end issued and outstanding shares net o...
fully_diluted_period_end_status                                          unavailable
Name: INR crore except per share and shares, dtype: object

status                                          unavailable
reason    A direct BSE closing price on the dated share ...
Name: Market-comparison status, dtype: object

,4.0%,5.0%,6.0%
10.0%,1337.15,1557.73,1888.60
12.0%,992.69,1102.64,1249.24
14.0%,786.23,849.97,929.64


## Refresh safely
Live scraping is intentionally not part of normal runs. To refresh, manually transcribe and verify official report rows in `data/raw/infosys_source_records.csv`, preserving source URL, page, period, basis, units and status. Run the script and tests afterwards. See `docs/unresolved_issues.md` for remaining review work.